# Lab Assignment 2 — Tutorial Demo
## Trigram Language Model, Add-k Smoothing and Perplexity

We build the complete pipeline on a **6-sentence toy corpus**, so every number is small enough
to check by hand.

---
## First: why a trigram?

A language model gives a probability to a sentence. The exact way to do that is the chain rule:

$$P(w_1 \dots w_n) = P(w_1) \cdot P(w_2 \mid w_1) \cdot P(w_3 \mid w_1, w_2) \cdots$$

The last term conditions on the **whole** sentence so far, and no corpus contains the same long
sentence twice. So we cut the history short — the **Markov assumption**:

| Model | Uses | Approximation |
|---|---|---|
| Bigram (last lab) | 1 previous word | $P(w_i \mid w_{i-1})$ |
| **Trigram (this lab)** | **2 previous words** | $P(w_i \mid w_{i-2}, w_{i-1})$ |

and we estimate it by counting:

$$P(w_3 \mid w_1, w_2) = \frac{\text{Count}(w_1, w_2, w_3)}{\text{Count}(w_1, w_2)}$$

Two words of context make the model **smarter** but also make the counts **sparser** — most
trigrams never appear at all. Everything else in this lab deals with that problem.

In [1]:
import math
from collections import Counter

corpus = [
    "I want to eat",
    "I want chinese food",
    "I want to eat chinese food",
    "I would like to know",
    "who am I",
    "I am here",
]
print(f"{len(corpus)} sentences")

6 sentences


---
# 1. Preprocessing and counting &nbsp; → Task 1

### Why **two** start tokens?

A trigram predicts a word from the two words before it. The first real word of a sentence has no
two words before it, so we invent them: pad with `<.s>` **twice**, and close with `<./s>` once.

    "I am here"  ->  ['<.s>', '<.s>', 'i', 'am', 'here', '<./s>']

Now the first word has a valid history: $P(\text{i} \mid \texttt{<.s>}, \texttt{<.s>})$.
With only one `<.s>` there would be no way to write that probability at all.

> Bigram needed one start token, trigram needs two. The rule is **n − 1**.

In [2]:
START, END = "<.s>", "<./s>"

def preprocess(sentence):
    """Lowercase, split, and pad for a TRIGRAM model."""
    return [START, START] + sentence.lower().split() + [END]

print(preprocess("I am here"))
print(preprocess("I want to eat"))

['<.s>', '<.s>', 'i', 'am', 'here', '<./s>']
['<.s>', '<.s>', 'i', 'want', 'to', 'eat', '<./s>']


### One `extract_ngrams` function for every `n`

We need unigrams, bigrams **and** trigrams, so write it once and pass `n` in. Slide a window of
width `n` along the tokens:

    tokens = [a, b, c, d]      n = 3
      i = 0  ->  (a, b, c)
      i = 1  ->  (b, c, d)     the last start position is len(tokens) - n

Return **tuples**, not lists — tuples can be used as dictionary keys, lists cannot.

In [3]:
def extract_ngrams(tokens, n):
    """Return the list of n-grams (as tuples) in a token list."""
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

toy = preprocess("I want to eat")
print("tokens  :", toy)
print("unigrams:", extract_ngrams(toy, 1))
print("bigrams :", extract_ngrams(toy, 2))
print("trigrams:", extract_ngrams(toy, 3))

tokens  : ['<.s>', '<.s>', 'i', 'want', 'to', 'eat', '<./s>']
unigrams: [('<.s>',), ('<.s>',), ('i',), ('want',), ('to',), ('eat',), ('<./s>',)]
bigrams : [('<.s>', '<.s>'), ('<.s>', 'i'), ('i', 'want'), ('want', 'to'), ('to', 'eat'), ('eat', '<./s>')]
trigrams: [('<.s>', '<.s>', 'i'), ('<.s>', 'i', 'want'), ('i', 'want', 'to'), ('want', 'to', 'eat'), ('to', 'eat', '<./s>')]


### Counting — and which table is the denominator

Look carefully at the formula. It needs **two different tables at the same time**:

$$P(w_3 \mid w_1, w_2) = \frac{\text{Count}(w_1, w_2, w_3)}{\text{Count}(w_1, w_2)}
\qquad \begin{array}{l}\leftarrow \text{a \textbf{trigram} count} \\ \leftarrow \text{a \textbf{bigram} count}\end{array}$$

The denominator of a trigram probability is a **bigram** count, because we are conditioning on
two words. The unigram counts are used for one thing only: the vocabulary size `V`.

In [ ]:
unigram_counts, bigram_counts, trigram_counts = Counter(), Counter(), Counter()

for sentence in corpus:
    tokens = preprocess(sentence)
    unigram_counts.update(extract_ngrams(tokens, 1))
    bigram_counts.update(extract_ngrams(tokens, 2))
    trigram_counts.update(extract_ngrams(tokens, 3))

V = len(unigram_counts)          # vocabulary size, needed for smoothing

print(f"V (unique tokens)      = {V}")
print(f"distinct trigrams seen = {len(trigram_counts)}")
print()
print(f"Count('i', 'want')        = {bigram_counts[('i', 'want')]}")
print(f"Count('i', 'want', 'to')  = {trigram_counts[('i', 'want', 'to')]}")
print(f"Count('i', 'want', 'a')   = {trigram_counts[('i', 'want', 'a')]}   <- never seen")

V (unique tokens)      = 14
distinct trigrams seen = 22

Count('i', 'want')        = 3
Count('i', 'want', 'to')  = 2
Count('i', 'want', 'a')   = 0   <- never seen


---
# 2. Trigram probability with Add-1 &nbsp; → Task 2

### The problem

Plain counting (MLE) gives probability **0** to every trigram that never occurred. A sentence
probability is a **product**, so a single zero makes the whole sentence impossible — and
$\log 0 = -\infty$.

### Add-1 (Laplace) smoothing

Pretend we saw every possible trigram **one extra time**:

$$P(w_3 \mid w_1, w_2) = \frac{\text{Count}(w_1, w_2, w_3) + 1}{\text{Count}(w_1, w_2) + V}$$

We added 1 to each of the `V` possible next words, so we added `V` in total — that is why the
denominator gets `+ V`. It keeps the probabilities adding up to 1.

In [5]:
def calculate_trigram_prob(trigram, trigram_counts, bigram_counts, V):
    """Add-1 smoothed P(w3 | w1, w2)."""
    history = trigram[:2]                        # (w1, w2) -> a BIGRAM key
    tri_count = trigram_counts.get(trigram, 0)
    hist_count = bigram_counts.get(history, 0)
    return (tri_count + 1) / (hist_count + V)


def mle_prob(trigram):
    """No smoothing, for comparison."""
    hist_count = bigram_counts.get(trigram[:2], 0)
    return 0.0 if hist_count == 0 else trigram_counts.get(trigram, 0) / hist_count


seen   = ('i', 'want', 'to')      # occurs twice
unseen = ('i', 'want', 'a')       # never occurs

for name, tg in [("SEEN  ", seen), ("UNSEEN", unseen)]:
    print(f"{name} {str(tg):<22} MLE = {mle_prob(tg):.4f}    "
          f"Add-1 = {calculate_trigram_prob(tg, trigram_counts, bigram_counts, V):.4f}")

print()
print("MLE gives the unseen trigram 0.0 -> the whole sentence collapses to 0.")
print("Add-1 gives it a small but usable value instead.")

SEEN   ('i', 'want', 'to')    MLE = 0.6667    Add-1 = 0.1765
UNSEEN ('i', 'want', 'a')     MLE = 0.0000    Add-1 = 0.0588

MLE gives the unseen trigram 0.0 -> the whole sentence collapses to 0.
Add-1 gives it a small but usable value instead.


---
# 3. Sentence log probability &nbsp; → Task 3

Multiplying many small probabilities gives a number too small for the computer to hold — this is
**underflow**. So we **add logarithms** instead of multiplying probabilities:

$$\log(P_1 \times P_2) = \log P_1 + \log P_2$$

We use $\log_2$ because the perplexity formula in the next section is base 2.

For `"i want food"` the padded tokens are `<.s> <.s> i want food <./s>`, which gives four
trigrams — one for each word we predict, including `<./s>`:

$$P(\text{i} \mid \texttt{<.s>},\texttt{<.s>}) \cdot P(\text{want} \mid \texttt{<.s>},\text{i})
\cdot P(\text{food} \mid \text{i},\text{want}) \cdot P(\texttt{<./s>} \mid \text{want},\text{food})$$

In [6]:
def calculate_sentence_log_prob_trigram(sentence, trigram_counts, bigram_counts, V):
    """Total log2 probability of a sentence."""
    tokens = preprocess(sentence)
    total = 0.0
    for trigram in extract_ngrams(tokens, 3):
        prob = calculate_trigram_prob(trigram, trigram_counts, bigram_counts, V)
        total += math.log(prob, 2)
    return total


s = "i want food"
print(f"the four trigrams of '{s}':\n")
for tg in extract_ngrams(preprocess(s), 3):
    p = calculate_trigram_prob(tg, trigram_counts, bigram_counts, V)
    print(f"   P({tg[2]!r:<8} | {tg[0]!r}, {tg[1]!r})".ljust(46) + f"= {p:.4f}")

L = calculate_sentence_log_prob_trigram(s, trigram_counts, bigram_counts, V)
print(f"\nsum of the log2 probabilities = {L:.4f}")

the four trigrams of 'i want food':

   P('i'      | '<.s>', '<.s>')               = 0.3000
   P('want'   | '<.s>', 'i')                  = 0.2105
   P('food'   | 'i', 'want')                  = 0.0588
   P('<./s>'  | 'want', 'food')               = 0.0714

sum of the log2 probabilities = -11.8797


---
# 4. Perplexity &nbsp; → Task 4

A log probability cannot be compared between sentences of different lengths, because a longer
sentence always scores lower just for having more terms. **Perplexity** divides that out:

$$PP(W) = P(w_1 \dots w_N)^{-1/N} \;=\; 2^{-L/N}$$

Both forms are the same thing; the second is the one we implement, because `L` is what section 3
gives us.

**What `N` counts:** the real words **plus** `<./s>`, but **not** the two `<.s>` pads — we
supplied those, we never predicted them. For `"i want food"`, `N = 4`.

**Lower perplexity is better.** Perplexity is roughly an *average number of choices*: a
perplexity of 5 means that at each word the model was about as unsure as if it were picking
between 5 words at random.

In [7]:
def calculate_perplexity(sentence, trigram_counts, bigram_counts, V):
    """Perplexity = 2 ** (-L / N)."""
    L = calculate_sentence_log_prob_trigram(sentence, trigram_counts, bigram_counts, V)
    N = len(sentence.lower().split()) + 1        # words + <./s>, NOT the <.s> pads
    return 2 ** (-L / N)


for s in ["i want to eat", "i want chinese food", "who am here"]:
    print(f"perplexity = {calculate_perplexity(s, trigram_counts, bigram_counts, V):7.2f}   '{s}'")

print()
print("The first two sentences are in the corpus  -> the model is not surprised.")
print("The last one is not                        -> the model is much more surprised.")
print(f"\nFor scale: a model that learned nothing would score {V} (our vocabulary size).")

perplexity =    5.21   'i want to eat'
perplexity =    5.57   'i want chinese food'
perplexity =    9.58   'who am here'

The first two sentences are in the corpus  -> the model is not surprised.
The last one is not                        -> the model is much more surprised.

For scale: a model that learned nothing would score 14 (our vocabulary size).


---
# 5. Add-k smoothing &nbsp; → Task 5

The `1` in Add-1 is not a magic number — it is just a choice, and on a small corpus it is usually
a **bad** one. Replace it with a tunable constant `k`:

$$P(w_3 \mid w_1, w_2) = \frac{\text{Count}(w_1, w_2, w_3) + k}{\text{Count}(w_1, w_2) + k \cdot V}$$

Note the denominator is `+ k·V`, **not** `+ k`: we gave `k` extra counts to each of `V` words, so
we handed out `k·V` in total.

Watch what Add-1 does to a trigram we have genuinely seen — it is worse than you would expect.

In [8]:
# Quick demo helper. It reads the counts straight from the cells above, so it does
# NOT have the shape your assignment asks for -- there you must pass the counts in
# as arguments and also handle the k = 0 (no smoothing) case yourself.
def addk(trigram, k):
    c = trigram_counts.get(trigram, 0)
    h = bigram_counts.get(trigram[:2], 0)
    return (c + k) / (h + k * V)


tg = ('i', 'want', 'to')
print(f"Count('i','want','to') = {trigram_counts[tg]},  Count('i','want') = {bigram_counts[tg[:2]]},  V = {V}\n")
print(f"  MLE       {trigram_counts[tg]}/{bigram_counts[tg[:2]]}            = {mle_prob(tg):.4f}")
print(f"  Add-1     (2+1)/(3+{V})   = {addk(tg, 1):.4f}")
print(f"  Add-0.1   (2+.1)/(3+{V*0.1:.1f}) = {addk(tg, 0.1):.4f}")
print("\nAdd-1 pushed a well-supported 0.67 down to 0.18, because it invented")
print(f"{V} fake counts to sit next to only 3 real ones. That is why we tune k.")

Count('i','want','to') = 2,  Count('i','want') = 3,  V = 14

  MLE       2/3            = 0.6667
  Add-1     (2+1)/(3+14)   = 0.1765
  Add-0.1   (2+.1)/(3+1.4) = 0.4773

Add-1 pushed a well-supported 0.67 down to 0.18, because it invented
14 fake counts to sit next to only 3 real ones. That is why we tune k.


---
# 6. Finding the best k &nbsp; → Task 6

`k` is a **hyperparameter**. We choose it the way we choose any hyperparameter: try a range of
values on a sentence the model was **not** trained on, and keep the one with the **lowest
perplexity**.

- **k too small** → trusts the counts, but punishes unseen trigrams very harshly
- **k too large** → everything is flattened towards uniform and the corpus stops mattering

The test sentence below uses only known words, but 3 of its 7 trigrams never appeared in training.

In [9]:
# Again a compact demo helper, not the shape your assignment asks for.
def pp_addk(sentence, k):
    L = sum(math.log(addk(tg, k), 2)
            for tg in extract_ngrams(preprocess(sentence), 3))
    return 2 ** (-L / (len(sentence.split()) + 1))


test_sentence = "i want to know chinese food"
k_values = [0.01, 0.1, 0.5, 1, 2, 5, 10]

print(f"Test sentence: '{test_sentence}'\n")
print(f"{'k':>6} | {'perplexity':>12}")
print("-" * 22)

results = {}
for k in k_values:
    results[k] = pp_addk(test_sentence, k)
    print(f"{k:>6} | {results[k]:>12.3f}")

best_k = min(results, key=results.get)
print("-" * 22)
print(f"\nBest k = {best_k}  (lowest perplexity = {results[best_k]:.3f})")
print()
print("Look at the shape of the column, not just the winner:")
print("  k = 0.01 -> 7.38   too small, the unseen trigrams are punished hard")
print("  k = 0.1  -> 5.29   the sweet spot")
print("  k = 10   -> 12.31  too large, the evidence is washed out")
print("\nThat U-shape is the whole reason we tune a hyperparameter.")

Test sentence: 'i want to know chinese food'

     k |   perplexity
----------------------
  0.01 |        7.377
   0.1 |        5.291
   0.5 |        6.462
     1 |        7.702
     2 |        9.222
     5 |       11.183
    10 |       12.313
----------------------

Best k = 0.1  (lowest perplexity = 5.291)

Look at the shape of the column, not just the winner:
  k = 0.01 -> 7.38   too small, the unseen trigrams are punished hard
  k = 0.1  -> 5.29   the sweet spot
  k = 10   -> 12.31  too large, the evidence is washed out

That U-shape is the whole reason we tune a hyperparameter.


---
## Before you go: what your assignment adds

Everything above is the foundation. Your assignment uses the same six steps, but pushes two of
them further.

**1. You will implement Add-k, not Add-1.** You saw both here; you write the general form.

**2. You will then build a better model than Add-k.** Here is the idea, and you write the code.

Add-k has one real weakness: it treats **every** unseen trigram exactly the same. Consider two
trigrams, neither in the corpus:

- `('want', 'to', 'eat')` — the pair `('to', 'eat')` is common, so this should still be likely
- `('credit', 'card', 'eat')` — nothing about this is plausible at any level

Add-k gives them the same tiny probability. It has no way to use the fact that a **shorter**
context was seen often.

The fix is called **linear interpolation**: instead of trusting the trigram estimate alone, take
a weighted average of all three models you already counted in section 1.

$$P(w_3 \mid w_1, w_2) = \lambda_1 P(w_3 \mid w_1, w_2) + \lambda_2 P(w_3 \mid w_2) + \lambda_3 P(w_3)$$

with $\lambda_1 + \lambda_2 + \lambda_3 = 1$. Instead of tuning one number `k`, you tune three
weights. The unigram term acts as a safety net so nothing is ever zero.

> **No code for this one.** You already have every count you need from section 1 — the
> assignment walks you through the rest.

---
## Summary — the six steps you implement

1. **Task 1** — pad with **two** `<.s>` and one `<./s>`; count unigrams, bigrams and trigrams; `V = len(unigram_counts)`
2. **Task 2** — Add-k: `(count + k) / (history count + k*V)`, where the history count is a **bigram** count
3. **Task 3** — **sum** the `log2` probabilities of the sentence's trigrams
4. **Task 4** — `2 ** (-L / N)`, where `N` = number of words + 1
5. **Task 5** — linear interpolation: mix the trigram, bigram and unigram estimates with weights
6. **Task 6** — tune `k`, tune the weights, and compare the two models

### Four mistakes that cost marks

- only **one** `<.s>` instead of two
- dividing by a **unigram** count instead of the **bigram** count
- writing `+ k` instead of `+ k*V` in the denominator
- counting the `<.s>` pads in `N` when computing perplexity